In [144]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
train = pd.read_csv("/kaggle/input/competitions/machine-translation-ioai/train.csv")
test = pd.read_csv("/kaggle/input/competitions/machine-translation-ioai/test.csv")

In [145]:
train

,data,label
0,den tjugofjärde 05 2049,24-05-2049
1,15/11/77,15-11-2077
2,sipsa'e 02 2049,14-02-2049
3,девятого 08 2049,09-08-2049
4,den femte 07 2077,05-07-2077
...,...,...
1090,den femte 02 2007,05-02-2007
1091,isipyuke 06 2007,26-06-2007
1092,le neuf mars 2007,09-03-2007
1093,am vier und zwanzigsten juni 2007,24-06-2007


In [146]:
test

,id,data
0,0,24 января 2007
1,1,le six mars 2049
2,2,le dix 05 2077
3,3,27 июня 2049
4,4,08 гыйнварда 2077
...,...,...
4671,4671,am fünfzehnten januar 2049
4672,4672,тугызынчы 05 2049
4673,4673,der achzehnte 02 2007
4674,4674,vierzehnter 12 2049


In [147]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32, padding_idx=pad_idx)
        self.rnn = nn.GRU(
            input_size=32,
            hidden_size=32,
            num_layers=4,
            bidirectional=True,
            batch_first=True
        )
        self.dropout = nn.Dropout(0.2)
        self.fc_hidden = nn.Linear(32 * 2, 32) 
        self.fc_output = nn.Linear(32 * 2, 32)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded)
        output = self.fc_output(output) 
        
        last_forward = hidden[-2, :, :]
        last_backward = hidden[-1, :, :]
        combined_hidden = torch.cat((last_forward, last_backward), dim=1)
        
        last_hidden = torch.tanh(self.fc_hidden(combined_hidden)) 
        last_hidden = last_hidden.unsqueeze(0).repeat(4, 1, 1)
        
        return output, last_hidden

In [153]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_a = nn.Linear(32, 32)
        self.U_a = nn.Linear(32, 32)
        self.v_a = nn.Linear(32, 1)

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        query = self.W_a(decoder_hidden).unsqueeze(1)
        keys = self.U_a(encoder_outputs)
        energy = torch.tanh(query + keys)
        scores = self.v_a(energy).squeeze(2)
        
        if src_mask is not None:
            scores = scores.masked_fill(src_mask, -1e4)
            
        attention_weights = F.softmax(scores, dim=1) 
        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)

        return context_vector.squeeze(1), attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32, padding_idx=pad_idx)
        self.rnn = nn.GRU(
            input_size=32 + 32, 
            hidden_size=32,
            num_layers=4,
            bidirectional=False, 
            batch_first=True
        )
        self.attention = Attention()
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x, hidden, encoder_outputs, src_mask):
        embedded = self.embedding(x)  
        context_vector, _ = self.attention(hidden[-1], encoder_outputs, src_mask)
        rnn_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=2)  
        output, hidden = self.rnn(rnn_input, hidden)  
        output = self.fc(output)  
        return output, hidden

In [156]:
import random
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = myvocab(train['data'])
target = myvocab(train['label']) 

num_epochs = 10
learning_rate = 0.001
max_seq_length = 40
teacher_forcing_ratio = 0.5

encoder = Encoder(data.n, pad_idx=data.vocab2idx['<PAD>']).to(device)
decoder = Decoder(target.n, pad_idx=target.vocab2idx['<PAD>']).to(device)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=target.vocab2idx['<PAD>'], reduction='sum')


Building Vocab: 100%|██████████| 1095/1095 [00:00<00:00, 1146471.01it/s]


In [178]:

for epoch in range(num_epochs):
    encoder.train()
    decoder.train()

    total_loss = 0
    
    progress_bar = tqdm(
        zip(train['data'], train['label']), 
        total=len(train['data']), 
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )
    
    for src_sentence, tgt_sentence in progress_bar:
        src_encoded = data.encode(src_sentence) + [data.vocab2idx['<EOS>']]
        tgt_encoded = [target.vocab2idx['<SOS>']] + target.encode(tgt_sentence) + [target.vocab2idx['<EOS>']]
        
        src_tensor = data.pad_sequences([src_encoded], max_length=max_seq_length).to(device)
        tgt_tensor = target.pad_sequences([tgt_encoded], max_length=max_seq_length).to(device)

        src_mask = (src_tensor == data.vocab2idx['<PAD>'])

        optimizer.zero_grad()

        encoder_output, last_hidden_state = encoder(src_tensor)
        decoder_input = tgt_tensor[:, 0:1]  
        loss = 0
        valid_steps = 0
        
        for t in range(1, tgt_tensor.size(1)):
            decoder_output, last_hidden_state = decoder(decoder_input, last_hidden_state, encoder_output, src_mask)
            
            step_target = tgt_tensor[:, t]
            loss += criterion(decoder_output.squeeze(1), step_target)
            
            if step_target.item() != target.vocab2idx['<PAD>']:
                valid_steps += 1
                
            use_teacher_forcing = random.random() < teacher_forcing_ratio
            
            if use_teacher_forcing:
                decoder_input = tgt_tensor[:, t:t+1]
            else:
                predicted_token = decoder_output.argmax(dim=-1)
                decoder_input = predicted_token 
            
            if step_target.item() == target.vocab2idx['<EOS>']:
                break

        if valid_steps > 0:
            loss = loss / valid_steps

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    epoch_loss = total_loss / len(train['data'])
    print(f'Summary -> Epoch {epoch + 1} Complete | Average Loss: {epoch_loss:.4f}\n')

Epoch 1/10: 100%|██████████| 1095/1095 [00:43<00:00, 25.34it/s, Loss=0.0059]


Summary -> Epoch 1 Complete | Average Loss: 0.0333



Epoch 2/10: 100%|██████████| 1095/1095 [00:43<00:00, 25.26it/s, Loss=0.0027]


Summary -> Epoch 2 Complete | Average Loss: 0.0263



Epoch 3/10: 100%|██████████| 1095/1095 [00:43<00:00, 25.33it/s, Loss=0.0016]


...

Summary -> Epoch 7 Complete | Average Loss: 0.0204



Epoch 8/10: 100%|██████████| 1095/1095 [00:44<00:00, 24.64it/s, Loss=0.0014]


Summary -> Epoch 8 Complete | Average Loss: 0.0188



Epoch 9/10:   7%|▋         | 79/1095 [00:03<00:41, 24.75it/s, Loss=0.0017]


KeyboardInterrupt: 

In [179]:
encoder.eval()
decoder.eval()

predictions = []

with torch.no_grad():
    for src_sentence in tqdm(test['data'], desc="Generating"):
        src_encoded = data.encode(src_sentence)
        src_tensor = data.pad_sequences([src_encoded], max_length=max_seq_length).to(device)
        src_mask = (src_tensor == data.vocab2idx['<PAD>'])

        encoder_output, last_hidden_state = encoder(src_tensor)
        
        decoder_input = torch.tensor([[target.vocab2idx['<SOS>']]], device=device)
        
        decoded_indices = []
        
        for _ in range(10):
            decoder_output, last_hidden_state = decoder(
                decoder_input, last_hidden_state, encoder_output, src_mask
            )
            
            predicted_token = decoder_output.argmax(dim=-1)
            decoded_indices.append(predicted_token.item())
            
            decoder_input = predicted_token
            
        predicted_chars = target.decode(decoded_indices)
        predicted_string = "".join(predicted_chars)
        predictions.append(predicted_string)

for i in range(min(10, len(test['data']))):
    print(f"Input: {test['data'][i]}")
    print(f"Output: {predictions[i]}")
    print("-" * 30)

Generating: 100%|██████████| 4676/4676 [00:35<00:00, 131.54it/s]

Input: 24 января 2007
Output: 24-01-2007
------------------------------
Input: le six mars 2049
Output: 18-03-2049
------------------------------
Input: le dix 05 2077
Output: 10-05-2077
------------------------------
Input: 27 июня 2049
Output: 27-06-2049
------------------------------
Input: 08 гыйнварда 2077
Output: 08-01-2077
------------------------------
Input: двадесет први 07 2007
Output: 21-07-2007
------------------------------
Input: isip samwol 2049
Output: 20-03-2049
------------------------------
Input: chil sa'wol 2007
Output: 07-04-2007
------------------------------
Input: 24 фебруара 2007
Output: 24-02-2007
------------------------------
Input: achzehnter 03 2077
Output: 18-03-2077
------------------------------


In [180]:
submission = pd.DataFrame({
    'id': test['id'],
    'label':predictions
})

In [181]:
submission.to_csv('submission.csv', index = False)